# Imports

In [1]:
import re
import optuna

import numpy as np
import pandas as pd
import skfuzzy as fuzz

import matplotlib.pyplot as plt

from boruta import BorutaPy
from sklearn import set_config
from category_encoders import TargetEncoder, OrdinalEncoder, CatBoostEncoder

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler, TargetEncoder, QuantileTransformer, StandardScaler, MinMaxScaler

## Utils

In [2]:
set_config(transform_output="pandas")

In [3]:
class ClusterFeatureGenerator(BaseEstimator, TransformerMixin):
    
    def __init__(self, n_clusters=5, cols=None, random_state=42, m=2.0):
        self.n_clusters = n_clusters
        self.cols = cols
        self.random_state = random_state
        self.m = m
        
        self.scaler = StandardScaler()
    
    def fit(self, X, y=None):
        X = X.copy()
        
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=np.number).columns
        else:
            self.cols_ = self.cols
        
        X_num = X[self.cols_].fillna(0)
        X_scaled = self.scaler.fit_transform(X_num)
        
        self.cntr_, self.u_, _, _, _, _, _ = fuzz.cluster.cmeans(
            X_scaled.T,
            c=self.n_clusters,
            m=self.m,
            error=1e-5,
            maxiter=1000,
            seed=self.random_state
        )
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        X_num = X[self.cols_].fillna(0)
        X_scaled = self.scaler.transform(X_num)
        
        u, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
            X_scaled.T,
            self.cntr_,
            m=self.m,
            error=1e-5,
            maxiter=1000
        )
        
        for i in range(self.n_clusters):
            X[f'fuzzy_{i}_{"_".join(self.cols_)}'] = u[i]
        
        return X

In [4]:
class AdaptiveFuzzyTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        columns=None,
        n_quantiles=1000,
        output_distribution="uniform",
        append=True,
        eps=1e-4,
        handle_degenerate="constant"  # "constant" ou "skip"
    ):
        self.columns = columns
        self.n_quantiles = n_quantiles
        self.output_distribution = output_distribution
        self.append = append
        self.eps = eps
        self.handle_degenerate = handle_degenerate

        self.transformers_ = {}
        self.bounds_ = {}
        self.is_degenerate_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        for col in self.columns:
            qt = QuantileTransformer(
                n_quantiles=min(self.n_quantiles, X.shape[0]),
                output_distribution=self.output_distribution
            )

            q = qt.fit_transform(X[[col]]).values.flatten()

            if np.std(q) < 1e-6:
                self.is_degenerate_[col] = True
                self.transformers_[col] = qt
                self.bounds_[col] = (0.25, 0.5, 0.75)  # dummy
                continue
            else:
                self.is_degenerate_[col] = False

            q25, q50, q75 = np.quantile(q, [0.25, 0.5, 0.75])

            if q25 >= q50:
                q25 = max(0, q50 - self.eps)

            if q75 <= q50:
                q75 = min(1, q50 + self.eps)

            if q25 >= q75:
                q25 = max(0, q50 - self.eps)
                q75 = min(1, q50 + self.eps)

            q25 = np.clip(q25, 0, 1)
            q50 = np.clip(q50, 0, 1)
            q75 = np.clip(q75, 0, 1)

            self.transformers_[col] = qt
            self.bounds_[col] = (q25, q50, q75)

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        fuzzy_features = []

        for col in self.columns:
            qt = self.transformers_[col]
            q = qt.transform(X[[col]]).values.flatten()

            if self.is_degenerate_[col]:
                if self.handle_degenerate == "skip":
                    continue

                low = np.zeros_like(q)
                medium = np.ones_like(q)
                high = np.zeros_like(q)

            else:
                q25, q50, q75 = self.bounds_[col]

                low = fuzz.trimf(q, [0, 0, q50])
                medium = fuzz.trimf(q, [q25, q50, q75])
                high = fuzz.trimf(q, [q50, 1, 1])

                low = np.nan_to_num(low)
                medium = np.nan_to_num(medium)
                high = np.nan_to_num(high)

            df_fuzzy = pd.DataFrame({
                f"{col}_low": low,
                f"{col}_medium": medium,
                f"{col}_high": high
            }, index=X.index)

            fuzzy_features.append(df_fuzzy)

        if not fuzzy_features:
            return X if self.append else pd.DataFrame(index=X.index)

        fuzzy_df = pd.concat(fuzzy_features, axis=1)

        if self.append:
            return pd.concat([X, fuzzy_df], axis=1)
        else:
            return fuzzy_df

In [5]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        X['lap_time_inv'] = 1 / (X['LapTime (s)'] + 1e-6)
        X['degradation_per_lap'] = X['Cumulative_Degradation'] / (X['LapNumber'] + 1e-6)
        
        X['position_norm'] = X.groupby('Race')['Position'].transform(lambda x: x / x.max())
        X['position_change_cum'] = X.groupby(['Race', 'Driver'])['Position_Change'].cumsum()
        
        X['is_pit_lap'] = (X['PitStop'] == 1).astype(int)
        X['laps_since_pit'] = X.groupby(['Race', 'Driver'])['is_pit_lap'].cumsum()
        
        X['tyre_life_ratio'] = X['TyreLife'] / (X['LapNumber'] + 1e-6)
        X['stint_progress'] = X.groupby(['Race', 'Driver', 'Stint']).cumcount()
        
        X['compound_tyre_life'] = (
            X['TyreLife'] * X['Compound'].astype('category').cat.codes
        )
        
        X['delta_x_tyre_life'] = X['LapTime_Delta'] * X['TyreLife']
        X['driver_mean_lap'] = X.groupby('Driver')['LapTime (s)'].transform('mean')
        
        X['race_progress_sin'] = np.sin(X['RaceProgress'] * np.pi)
        
        X = X.replace([np.inf, -np.inf], np.nan)
        
        return X

In [6]:
mapping = [
    {
        'col': 'Compound', 
        'mapping': {
            'SOFT': 1,
            'MEDIUM': 2,
            'HARD': 3,
            'INTERMEDIATE': 4,
            'WET': 5
        }
    }
]

ordinal_encoder = Pipeline([('ordinal', OrdinalEncoder(mapping=mapping)), ('minmax', MinMaxScaler())])

In [7]:
ct = ColumnTransformer([
    (
        'ordinal_encoder', 
        ordinal_encoder, 
        ['Compound']
    ),
    (
        'target_encoder', 
        TargetEncoder(), 
        ['Driver', 'Compound', 'Race']
    ),
    (
        'catboost_encoder', 
        CatBoostEncoder(), 
        ['Driver', 'Compound', 'Race']
    ),
    (
        'standard_scaler', 
        StandardScaler(), 
        ['LapNumber', 'Position', 'RaceProgress', 'Year', 'position_norm', 'race_progress_sin']
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'Position_Change', 'Cumulative_Degradation', 'LapTime_Delta', 
            'LapTime (s)', 'Stint', 'driver_mean_lap', 'TyreLife', 'delta_x_tyre_life', 
            'compound_tyre_life', 'stint_progress', 'tyre_life_ratio', 'degradation_per_lap', 
            'position_change_cum', 'laps_since_pit', 'lap_time_inv'
        ]
    ),
], remainder="passthrough", n_jobs=-1)

In [8]:
def clean_columns(df):
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]', '', col.replace(' ', '_')).lower()
        for col in df.columns
    ]
    return df

# Loading Dataset

In [9]:
df_train = pd.read_csv('../data/train.csv', index_col='id')
df_test = pd.read_csv('../data/test.csv', index_col='id')

In [10]:
df_train.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [11]:
df_train.tail()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,,,,,,,,,,,,,,,
439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0
439139,D714,HARD,Miami Grand Prix,2023,0,49,2,32.0,1,91.691,-0.123,-15.828,0.859649,0.0,0.0


## Columns

In [12]:
df_train.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='str')

In [13]:
df_test.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change'],
      dtype='str')

## Data Dimension

In [14]:
df_train.shape

(439140, 15)

In [15]:
df_test.shape

(188165, 14)

## Check NA

In [16]:
df_train.isna().mean()

Driver                    0.0
Compound                  0.0
Race                      0.0
Year                      0.0
PitStop                   0.0
LapNumber                 0.0
Stint                     0.0
TyreLife                  0.0
Position                  0.0
LapTime (s)               0.0
LapTime_Delta             0.0
Cumulative_Degradation    0.0
RaceProgress              0.0
Position_Change           0.0
PitNextLap                0.0
dtype: float64

In [17]:
df_test.isna().mean()

Driver                    0.0
Compound                  0.0
Race                      0.0
Year                      0.0
PitStop                   0.0
LapNumber                 0.0
Stint                     0.0
TyreLife                  0.0
Position                  0.0
LapTime (s)               0.0
LapTime_Delta             0.0
Cumulative_Degradation    0.0
RaceProgress              0.0
Position_Change           0.0
dtype: float64

## Splitting X and y

In [18]:
X_train = df_train.drop(['PitNextLap'], axis=1)
y_train = df_train.loc[:, ['PitNextLap']].astype(int)

X_test = df_test.copy()

# Feature Engineering

## Fuzzy Logic

In [19]:
fuzzy_transform = AdaptiveFuzzyTransformer(columns=['LapNumber', 'Stint', 'Position', 'LapTime (s)', 'RaceProgress'])

X_train = fuzzy_transform.fit_transform(X_train)
X_test = fuzzy_transform.transform(X_test)

## Clustering Feature Engineering

In [20]:
clustering_lap_time = ClusterFeatureGenerator(n_clusters=3, cols=['LapTime (s)', 'LapTime_Delta'])

X_train = clustering_lap_time.fit_transform(X_train)
X_test = clustering_lap_time.transform(X_test)

In [21]:
clustering_tyre_degradation = ClusterFeatureGenerator(n_clusters=3, cols=['TyreLife', 'Cumulative_Degradation'])

X_train = clustering_tyre_degradation.fit_transform(X_train)
X_test = clustering_tyre_degradation.transform(X_test)

In [22]:
clustering_position = ClusterFeatureGenerator(n_clusters=3, cols=['Position', 'Position_Change'])

X_train = clustering_position.fit_transform(X_train)
X_test = clustering_position.transform(X_test)

In [23]:
clustering_race_lap = ClusterFeatureGenerator(n_clusters=3, cols=['RaceProgress', 'LapNumber'])

X_train = clustering_race_lap.fit_transform(X_train)
X_test = clustering_race_lap.transform(X_test)

## Feature Engineering Transform

In [24]:
fet = FeatureEngineeringTransformer()

X_train = fet.fit_transform(X_train)
X_test = fet.transform(X_test)

## Encoding & Scalers

In [25]:
X_train = ct.fit_transform(X_train, y_train['PitNextLap'])
X_test = ct.transform(X_test)

## Clean Column Names

In [26]:
X_train = clean_columns(X_train)
X_test = clean_columns(X_test)

## Feature Selection

In [28]:
rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5)
feat_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=1)
feat_selector.fit(X_train, y_train.PitNextLap)

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	57
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	56
Tentative: 	1
Rejected: 	0
Iteration: 	9 / 100
Confirmed: 	56
Tentative: 	1
Rejected: 	0
Iteration: 	10 / 100
Confirmed: 	56
Tentative: 	1
Rejected: 	0
Iteration: 	11 / 100
Confirmed: 	56
Tentative: 	1
Rejected: 	0
Iteration: 	12 / 100
Confirmed: 	57
Tentative: 	0
Rejected: 	0


BorutaPy finished running.

Iteration: 	13 / 100
Confirmed: 	57
Tentative: 	0
Rejected: 	0


BorutaPy(estimator=RandomForestClassifier(class_weight='balanced', max_depth=5,
                                          n_estimators=213, n_jobs=-1,
                                          random_state=RandomState(MT19937) at 0x7F74BF1B8340),
         n_estimators='auto',
         random_state=RandomState(MT19937) at 0x7F74BF1B8340, verbose=2)

In [29]:
X_train.loc[:, feat_selector.support_]

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,catboost_encoder__driver,catboost_encoder__compound,catboost_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,...,remainder__fuzzy_0_tyrelife_cumulative_degradation,remainder__fuzzy_1_tyrelife_cumulative_degradation,remainder__fuzzy_2_tyrelife_cumulative_degradation,remainder__fuzzy_0_position_position_change,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__is_pit_lap
id,,,,,,,,,,,,,,,,,,,,,
0,0.50,0.202715,0.327274,0.153444,0.198982,0.198982,0.198982,1.585901,-0.308849,1.487008,...,0.123817,0.679020,0.197163,0.468046,0.440081,0.091874,0.000549,0.997580,0.001872,0
1,0.50,0.193796,0.327551,0.174983,0.198982,0.599491,0.198982,0.229628,-1.066602,0.033534,...,0.323123,0.273053,0.403825,0.164128,0.695050,0.140823,0.015175,0.007697,0.977128,1
2,0.50,0.225781,0.327893,0.187224,0.198982,0.399661,0.198982,2.116616,0.638343,1.902200,...,0.156952,0.372624,0.470424,0.913883,0.048476,0.037640,0.024077,0.912645,0.063278,0
3,0.25,0.189556,0.100630,0.145107,0.198982,0.198982,0.198982,-1.244581,-0.498287,-1.029455,...,0.897768,0.024037,0.078195,0.188322,0.749531,0.062147,0.950726,0.010153,0.039121,0
4,0.50,0.188831,0.327551,0.215101,0.198982,0.549746,0.198982,0.170660,-1.445478,0.092589,...,0.999852,0.000027,0.000121,0.103062,0.853629,0.043309,0.009589,0.004844,0.985566,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,0.25,0.198980,0.100907,0.104374,0.198982,0.101133,0.103580,1.526932,1.396096,2.060938,...,0.932580,0.010117,0.057303,0.589641,0.087787,0.322572,0.016454,0.936647,0.046898,0
439136,0.25,0.198983,0.101480,0.103614,0.198982,0.101133,0.103574,1.526932,-1.634917,2.060938,...,0.969289,0.006513,0.024198,0.077457,0.877420,0.045123,0.016454,0.936647,0.046898,0
439137,0.25,0.198983,0.100630,0.103162,0.198982,0.101132,0.103569,1.526932,-1.634917,2.387294,...,0.071467,0.122789,0.805744,0.077457,0.877420,0.045123,0.036392,0.869713,0.093895,0


# Exporting Datasets

In [30]:
X_train.loc[:, feat_selector.support_].to_parquet('../data/X_train.parquet')
X_test.loc[:, feat_selector.support_].to_parquet('../data/X_test.parquet')

y_train.to_parquet('../data/y_train.parquet')